<a href="https://colab.research.google.com/github/ansarmuhammad/fine-tuning/blob/main/eval_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#name=baseline_evaluator_improved.py url=https://github.com/ansarmuhammad/fine-tuning
#!/usr/bin/env python3
"""
Improved DeepEval Baseline Evaluator for Car Repair Q&A Model

Key improvements:
- Proper logging instead of print statements
- Config file support (YAML/JSON)
- Data validation and error handling
- Batch generation for better GPU utilization
- Async metric scoring
- Reusable classes and functions
- Better resource management
- Reproducibility (seeding)
- HTML report generation
"""

import os
import sys
import json
import logging
import time
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple
from collections import defaultdict
import warnings

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

!pip install litellm
import litellm
from tqdm.auto import tqdm
from google.colab import userdata # Added for Google API key

from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import AnswerRelevancyMetric, GEval, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# ============================================================
# SETUP LOGGING
# ============================================================

def setup_logging(log_dir: Path, level: int = logging.INFO) -> logging.Logger:
    """Configure structured logging."""
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / f"baseline_{time.strftime('%Y%m%d_%H%M%S')}.log"

    logger = logging.getLogger(__name__)
    logger.setLevel(level)

    # File handler
    fh = logging.FileHandler(log_file)
    fh.setLevel(level)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(level)

    # Format
    formatter = logging.Formatter(
        '%(asctime)s | %(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    return logger

# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    """Configuration for baseline evaluation."""

    # Model
    model_id: str = "unsloth/gemma-4-E2B-it"
    torch_dtype: str = "bfloat16"

    # Data
    data_dir: Path = field(default_factory=lambda: Path(__file__).parent / "data")
    train_file: str = "train.json"
    test_file: str = "test.json"

    # Evaluation
    eval_n_samples: Optional[int] = 10  # None = all
    batch_size: int = 4  # For generation
    async_scoring: bool = True

    # Generation
    max_new_tokens: int = 256
    max_input_length: int = 1024
    seed: int = 42

    # Judge (Gemini via LiteLLM)
    eval_model: str = "gemini/gemini-1.5-flash-latest" # Changed to Gemini model
    # ollama_base_url removed as it's specific to Ollama
    eval_threshold: float = 0.7
    judge_max_tokens: int = 1024
    judge_timeout: int = 30
    judge_retries: int = 5

    # Output
    base_dir: Path = field(default_factory=lambda: Path(__file__).parent / "baseline_output")

    def validate(self, logger: logging.Logger) -> None:
        """Validate config."""
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

        train_file = self.data_dir / self.train_file
        test_file = self.data_dir / self.test_file

        if not train_file.exists():
            raise FileNotFoundError(f"Training file not found: {train_file}")
        if not test_file.exists():
            raise FileNotFoundError(f"Test file not found: {test_file}")

        if self.eval_n_samples is not None and self.eval_n_samples <= 0:
            raise ValueError(f"eval_n_samples must be positive, got {self.eval_n_samples}")

        logger.info("✓ Config validated")

    @classmethod
    def from_file(cls, config_path: Path) -> "Config":
        """Load config from JSON or YAML."""
        if config_path.suffix == ".json":
            with open(config_path) as f:
                data = json.load(f)
        elif config_path.suffix in [".yaml", ".yml"]:
            try:
                import yaml
                with open(config_path) as f:
                    data = yaml.safe_load(f)
            except ImportError:
                raise ImportError("PyYAML required for YAML config. Install: pip install pyyaml")
        else:
            raise ValueError(f"Unsupported config format: {config_path.suffix}")

        return cls(**data)


# ============================================================
# LITELLM JUDGE (IMPROVED)
# ============================================================

class LiteLLMJudge(DeepEvalBaseLLM):
    """Local Ollama judge via LiteLLM with better error handling."""

    def __init__(
        self,
        model_name: str,
        # api_base removed as it's not applicable for hosted Gemini via LiteLLM
        max_tokens: int = 1024,
        timeout: int = 30,
        retries: int = 5,
        logger: Optional[logging.Logger] = None,
    ):
        self.model_name = model_name
        # self.api_base removed
        self.max_tokens = max_tokens
        self.timeout = timeout
        self.retries = retries
        self.logger = logger or logging.getLogger(__name__)

    def _call(self, prompt: str, schema: Optional[Any] = None) -> Any:
        """Call judge with retries and backoff."""
        kwargs = {
            "model": self.model_name,
            "messages": [{"role": "user", "content": prompt}],
            # api_base removed from kwargs
            "temperature": 0,
            "max_tokens": self.max_tokens,
            "timeout": self.timeout,
        }
        if schema is not None:
            kwargs["format"] = "json"

        last_err = None
        for attempt in range(self.retries):
            try:
                resp = litellm.completion(**kwargs)
                text = resp.choices[0].message.content.strip()

                # Clean markdown code blocks
                if text.startswith("```"):
                    parts = text.split("```")
                    if len(parts) >= 2:
                        text = parts[1]
                        if text.startswith("json"):
                            text = text[4:]
                        text = text.strip()

                if schema is not None:
                    try:
                        return schema.model_validate_json(text)
                    except Exception as pe:
                        last_err = pe
                        if attempt < self.retries - 1:
                            kwargs["max_tokens"] = min(kwargs["max_tokens"] * 2, 4096)
                            continue
                        raise

                return text

            except Exception as e:
                last_err = e
                msg = str(e).lower()

                if any(x in msg for x in ["503", "unavailable", "connection", "timeout"]):
                    wait = min(5 * (2 ** attempt), 30)
                    self.logger.warning(f"LLM service unavailable, retry {attempt+1}/{self.retries} in {wait}s") # Changed log message
                    time.sleep(wait)
                    continue

                if "validation" in msg or "json" in msg:
                    if attempt < self.retries - 1:
                        kwargs["max_tokens"] = min(kwargs["max_tokens"] * 2, 4096)
                        self.logger.debug(f"JSON validation error, retrying with {kwargs['max_tokens']} tokens")
                        continue

                raise

        raise RuntimeError(f"Judge failed after {self.retries} retries: {last_err}")

    def load_model(self) -> str:
        return self.model_name

    def generate(self, prompt: str, schema: Optional[Any] = None) -> Any:
        return self._call(prompt, schema)

    async def a_generate(self, prompt: str, schema: Optional[Any] = None) -> Any:
        return self._call(prompt, schema)

    def generate_raw_response(self, prompt: str, top_logprobs: Optional[Any] = None, schema: Optional[Any] = None) -> Tuple[Any, float]:
        text = self._call(prompt, schema)

        class _FakeChoice:
            def __init__(self, t):
                self.message = type("M", (), {"content": t})

        class _FakeResp:
            def __init__(self, t):
                self.choices = [_FakeChoice(t)]
                self.usage = type("U", (), {
                    "total_tokens": 0,
                    "prompt_tokens": 0,
                    "completion_tokens": 0
                })

        return _FakeResp(text if isinstance(text, str) else str(text)), 0.0

    async def a_generate_raw_response(self, prompt: str, top_logprobs: Optional[Any] = None, schema: Optional[Any] = None) -> Tuple[Any, float]:
        return self.generate_raw_response(prompt, top_logprobs, schema)

    def get_model_name(self) -> str:
        return self.model_name


# ============================================================
# DATA UTILITIES
# ============================================================

def load_json_data(filepath: Path) -> List[Dict[str, Any]]:
    """Load JSON data with error handling."""
    try:
        with open(filepath) as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError(f"Expected list, got {type(data)}")
        return data
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {filepath}: {e}")
    except Exception as e:
        raise RuntimeError(f"Failed to load {filepath}: {e}")


def find_key(item: Dict, candidates: List[str]) -> Optional[str]:
    """Find first non-empty key from candidates."""
    for k in candidates:
        if k in item and item[k]:
            return k
    return None


def detect_schema(data: List[Dict[str, Any]]) -> Tuple[Optional[str], Optional[str], Optional[str], bool]:
    """Auto-detect question, answer, context keys."""
    if not data:
        raise ValueError("Empty dataset")

    Q_CANDIDATES = ['question', 'input', 'query', 'prompt', 'instruction']
    A_CANDIDATES = ['answer', 'expected_answer', 'output', 'response', 'completion']
    C_CANDIDATES = ['context', 'retrieval_context', 'reference', 'passage', 'background']

    sample = data[0]

    q_key = find_key(sample, Q_CANDIDATES)
    a_key = find_key(sample, A_CANDIDATES)
    c_key = find_key(sample, C_CANDIDATES)

    if not q_key or not a_key:
        raise ValueError(f"Missing question/answer fields. Available: {list(sample.keys())}")

    # Check if context is usable (present in 80%+ of samples)
    has_context = bool(c_key) and sum(1 for it in data if it.get(c_key)) >= len(data) * 0.8

    return q_key, a_key, c_key, has_context


def to_str(value: Any) -> str:
    """Convert any value to string."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return "\n".join(str(item).strip() for item in value if item)
    return str(value)


# ============================================================
# BASELINE EVALUATOR (MAIN CLASS)
# ============================================================

class BaselineEvaluator:
    """End-to-end baseline evaluation pipeline."""

    def __init__(self, config: Config, logger: logging.Logger):
        self.config = config
        self.logger = logger

        # Setup output directories
        self.base_dir = config.base_dir
        self.results_dir = self.base_dir / "results"
        self.logs_dir = self.base_dir / "logs"
        self.artifacts_dir = self.base_dir / "artifacts"

        for d in [self.base_dir, self.results_dir, self.logs_dir, self.artifacts_dir]:
            d.mkdir(parents=True, exist_ok=True)

        # Output files
        self.answers_file = self.results_dir / "baseline_answers.json"
        self.scores_file = self.results_dir / "baseline_scores.json"
        self.checkpoint_file = self.results_dir / "checkpoint.jsonl"
        self.report_file = self.results_dir / "report.html"

        # State
        self.model = None
        self.tokenizer = None
        self.judge = None
        self.metrics = None
        self.data = None
        self.eval_data = None
        self.baseline_results = None
        self.test_cases = None
        self.scores_by_metric = defaultdict(list)
        self.per_case = []
        self.failed_count = 0

        # Schema
        self.q_key = None
        self.a_key = None
        self.c_key = None
        self.has_context = False

    def run(self) -> Dict[str, Any]:
        """Execute full pipeline."""
        try:
            self.logger.info("=" * 70)
            self.logger.info("BASELINE EVALUATION PIPELINE")
            self.logger.info("=" * 70)

            self.config.validate(self.logger)
            self._set_seed()
            self._check_environment()
            self._load_data()
            self._detect_schema()
            self._setup_judge()
            self._load_model()
            self._generate_answers()
            self._score_answers()
            self._generate_report()

            self.logger.info("=" * 70)
            self.logger.info("✓ PIPELINE COMPLETE")
            self.logger.info("=" * 70)

            return self._get_summary()

        except Exception as e:
            self.logger.error(f"Pipeline failed: {e}", exc_info=True)
            raise

    def _set_seed(self) -> None:
        """Set random seeds for reproducibility."""
        torch.manual_seed(self.config.seed)
        np.random.seed(self.config.seed)
        self.logger.info(f"Random seed: {self.config.seed}")

    def _check_environment(self) -> None:
        """Check GPU and environment."""
        self.logger.info(f"Python: {sys.version.split()[0]}")
        self.logger.info(f"PyTorch: {torch.__version__}")

        if torch.cuda.is_available():
            self.device = "cuda"
            n_gpu = torch.cuda.device_count()
            self.logger.info(f"CUDA available: {n_gpu} GPU(s)")
            for i in range(n_gpu):
                name = torch.cuda.get_device_name(i)
                vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
                self.logger.info(f"  GPU {i}: {name} ({vram:.1f} GB)")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            self.device = "mps"
            self.logger.info("Using Apple Metal Performance Shaders (MPS)")
        else:
            self.device = "cpu"
            self.logger.warning("No GPU available, using CPU (very slow)")

    def _load_data(self) -> None:
        """Load train and test data."""
        self.logger.info(f"Loading data from {self.config.data_dir}")

        train_path = self.config.data_dir / self.config.train_file
        test_path = self.config.data_dir / self.config.test_file

        train_data = load_json_data(train_path)
        test_data = load_json_data(test_path)

        self.logger.info(f"Train samples: {len(train_data)}")
        self.logger.info(f"Test samples: {len(test_data)}")

        # Cap to eval_n_samples
        if self.config.eval_n_samples:
            self.eval_data = test_data[:self.config.eval_n_samples]
        else:
            self.eval_data = test_data

        self.logger.info(f"Eval samples: {len(self.eval_data)}")

    def _detect_schema(self) -> None:
        """Detect question/answer/context field names."""
        self.q_key, self.a_key, self.c_key, self.has_context = detect_schema(self.eval_data)

        self.logger.info(f"Question field: '{self.q_key}'")
        self.logger.info(f"Answer field: '{self.a_key}'")
        self.logger.info(f"Context field: '{self.c_key}' (usable={self.has_context})")

    def _setup_judge(self) -> None:
        """Initialize judge and metrics."""
        self.logger.info("Setting up judge...")

        # Get Gemini API Key from Colab secrets
        try:
            GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
            litellm.api_key = GOOGLE_API_KEY # LiteLLM uses this for Gemini
            self.logger.info("✓ Google API Key loaded for Gemini")
        except Exception as e:
            raise ValueError("GOOGLE_API_KEY not found in Colab secrets. Please set it.") from e

        # Build judge
        self.judge = LiteLLMJudge(
            model_name=self.config.eval_model,
            # api_base is removed as it's not applicable for hosted Gemini via LiteLLM
            max_tokens=self.config.judge_max_tokens,
            timeout=self.config.judge_timeout,
            retries=self.config.judge_retries,
            logger=self.logger,
        )

        # Smoke test
        try:
            probe = self.judge.generate("Reply with exactly one word: OK")
            self.logger.info(f"✓ Judge working (response: {str(probe)[:50]})")
        except Exception as e:
            raise RuntimeError(f"Judge smoke test failed: {e}")

        # Build metrics
        self.metrics = [
            AnswerRelevancyMetric(
                threshold=self.config.eval_threshold,
                model=self.judge,
                async_mode=self.config.async_scoring,
            ),
            GEval(
                name="Correctness",
                criteria=(
                    "Evaluate whether the actual_output is a factually correct, technically accurate "
                    "answer to the input question, using expected_output as ground truth. For car repair, "
                    "check: correct parts/components, correct diagnostic reasoning, correct procedures, "
                    "safe advice. Heavily penalize vague, generic, or incorrect technical claims."
                ),
                evaluation_params=[
                    LLMTestCaseParams.INPUT,
                    LLMTestCaseParams.ACTUAL_OUTPUT,
                    LLMTestCaseParams.EXPECTED_OUTPUT,
                ],
                threshold=self.config.eval_threshold,
                model=self.judge,
                async_mode=self.config.async_scoring,
            ),
        ]

        if self.has_context:
            self.metrics += [
                FaithfulnessMetric(
                    threshold=self.config.eval_threshold,
                    model=self.judge,
                    async_mode=self.config.async_scoring,
                ),
                ContextualRelevancyMetric(
                    threshold=self.config.eval_threshold,
                    model=self.judge,
                    async_mode=self.config.async_scoring,
                ),
            ]

        self.logger.info(f"Metrics: {[m.name if hasattr(m, 'name') else m.__class__.__name__ for m in self.metrics]}")

    def _load_model(self) -> None:
        """Load LLM model."""
        self.logger.info(f"Loading model: {self.config.model_id}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        hf_token = os.environ.get("HF_TOKEN", "")

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.config.model_id,
                token=hf_token or None,
            )

            torch_dtype = getattr(torch, self.config.torch_dtype)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.config.model_id,
                torch_dtype=torch_dtype,
                device_map="auto",
                token=hf_token or None,
            )

            self.model.eval()

            self.logger.info(f"✓ Model loaded on device: {self.device}")

            if torch.cuda.is_available():
                for i in range(torch.cuda.device_count()):
                    name = torch.cuda.get_device_name(i)
                    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
                    self.logger.info(f"  GPU {i}: {name} ({vram:.1f} GB)")

        except Exception as e:
            raise RuntimeError(f"Failed to load model: {e}")

    def _generate_answers(self) -> None:
        """Generate answers for all eval samples."""
        self.logger.info(f"Generating answers for {len(self.eval_data)} samples...")

        SYSTEM_PROMPT = (
            "You are an expert car repair assistant. Answer the user's question concisely "
            "and accurately. Be technically precise about parts, diagnostics, and procedures."
        )

        def build_prompt(question: str) -> str:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ]
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

        @torch.no_grad()
        def generate(question: str) -> str:
            prompt = build_prompt(question)
            inputs = self.tokenizer(
                text=prompt,
                return_tensors="pt",
                truncation=True,
                max_length=self.config.max_input_length,
            ).to(self.model.device)

            out = self.model.generate(
                **inputs,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

            new_tokens = out[0][inputs['input_ids'].shape[1]:]
            return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        # Smoke test
        self.logger.info("Smoke test on first sample...")
        q0 = self.eval_data[0][self.q_key]
        a0 = generate(q0)
        self.logger.info(f"  Question: {q0[:100]}...")
        self.logger.info(f"  Generated: {a0[:200]}...")

        # Generate all
        self.baseline_results = []
        for item in tqdm(self.eval_data, desc="Generating"):
            q = item[self.q_key]
            expected = item[self.a_key]
            ctx = item.get(self.c_key, "") if self.c_key else ""
            gen = generate(q)

            self.baseline_results.append({
                "question": q,
                "expected_answer": expected,
                "generated_answer": gen,
                "context": ctx,
            })

        # Save
        with open(self.answers_file, 'w') as f:
            json.dump(self.baseline_results, f, indent=2, ensure_ascii=False)

        self.logger.info(f"✓ Saved {len(self.baseline_results)} answers → {self.answers_file}")

    def _score_answers(self) -> None:
        """Score answers using metrics."""
        self.logger.info("Building test cases...")

        self.test_cases = []
        for r in self.baseline_results:
            ctx_raw = r.get("context", "")
            ctx_str = to_str(ctx_raw)
            ctx_list = [ctx_str] if (self.has_context and ctx_str) else None

            self.test_cases.append(LLMTestCase(
                input=to_str(r["question"]),
                actual_output=to_str(r["generated_answer"]),
                expected_output=to_str(r["expected_answer"]),
                retrieval_context=ctx_list,
            ))

        total_calls = len(self.test_cases) * len(self.metrics)
        est_min = (total_calls * 6.0) / 60
        self.logger.info(f"Scoring: {len(self.test_cases)} cases × {len(self.metrics)} metrics = {total_calls} calls (~{est_min:.0f} min)")

        # Resume from checkpoint
        completed_indices = set()
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file) as f:
                for line in f:
                    try:
                        entry = json.loads(line)
                        completed_indices.add(entry["idx"])
                        self.per_case.append(entry)
                        for k, v in entry.get("scores", {}).items():
                            if isinstance(v, (int, float)):
                                self.scores_by_metric[k].append(v)
                    except:
                        pass

            if completed_indices:
                self.logger.info(f"Resuming from checkpoint: {len(completed_indices)} cases already scored")

        # Score
        with open(self.checkpoint_file, 'a') as ckpt_file:
            for i, tc in enumerate(tqdm(self.test_cases, desc="Scoring")):
                if i in completed_indices:
                    continue

                case_scores = {}
                case_failed = False

                for metric in self.metrics:
                    mname = getattr(metric, 'name', None) or metric.__class__.__name__
                    try:
                        metric.measure(tc)
                        score = getattr(metric, 'score', None)
                        if score is not None:
                            self.scores_by_metric[mname].append(score)
                            case_scores[mname] = round(score, 3)
                    except Exception as e:
                        self.logger.warning(f"Case {i}, metric {mname}: {str(e)[:100]}")
                        case_failed = True

                if case_failed and not case_scores:
                    self.failed_count += 1

                entry = {
                    "idx": i,
                    "question": to_str(self.baseline_results[i]["question"])[:150],
                    "scores": case_scores,
                }
                self.per_case.append(entry)

                ckpt_file.write(json.dumps(entry) + "\n")
                ckpt_file.flush()

        # Log results
        self.logger.info("\n" + "=" * 70)
        self.logger.info(f"BASELINE SCORES — {self.config.model_id}")
        self.logger.info("=" * 70)

        for name, vals in self.scores_by_metric.items():
            if not vals:
                continue

            avg = sum(vals) / len(vals)
            std = np.std(vals) if len(vals) > 1 else 0
            pass_rate = sum(1 for v in vals if v >= self.config.eval_threshold) / len(vals)

            self.logger.info(
                f"  {name:35s}  avg={avg:.3f}\u00B1{std:.3f}  pass_rate={pass_rate:.1%}  (n={len(vals)})"
            )

        if self.failed_count:
            self.logger.warning(f"  \u2622 {self.failed_count} cases failed scoring")

        self.logger.info("=" * 70)

    def _generate_report(self) -> None:
        """Generate HTML report."""
        self.logger.info(f"Generating report → {self.report_file}")

        summary = self._get_summary()

        html = f"""
        <html>
        <head>
            <title>Baseline Evaluation Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                table {{ border-collapse: collapse; width: 100%; }}
                th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
                th {{ background-color: #4CAF50; color: white; }}
                tr:nth-child(even) {{ background-color: #f2f2f2; }}
            </style>
        </head>
        <body>
            <h1>Baseline Evaluation Report</h1>

            <h2>Configuration</h2>
            <table>
                <tr><td>Model</td><td>{summary['model']}</td></tr>
                <tr><td>Dataset</td><td>{summary['dataset']}</td></tr>
                <tr><td>Judge</td><td>{summary['judge']}</td></tr>
                <tr><td>Samples</td><td>{summary['n_samples']}</td></tr>
                <tr><td>Threshold</td><td>{summary['threshold']}</td></tr>
            </table>

            <h2>Metrics</h2>
            <table>
                <tr>
                    <th>Metric</th>
                    <th>Avg Score</th>
                    <th>Pass Rate</th>
                    <th>Samples</th>
                </tr>
        """

        for name, stats in summary['metrics'].items():
            html += f"""
                <tr>
                    <td>{name}</td>
                    <td>{stats['avg_score']:.3f}</td>
                    <td>{stats['pass_rate']:.1%}</td>
                    <td>{stats['n']}</td>
                </tr>
            """

        html += """
            </table>
        </body>
        </html>
        """

        with open(self.report_file, 'w') as f:
            f.write(html)

    def _get_summary() -> Dict[str, Any]:
        """Get results summary."""
        summary = {
            "model": self.config.model_id,
            "dataset": str(self.config.data_dir / self.config.test_file),
            "judge": self.config.eval_model,
            "n_samples": len(self.test_cases) if self.test_cases else 0,
            "n_failed": self.failed_count,
            "threshold": self.config.eval_threshold,
            "metrics": {},
        }

        for name, vals in self.scores_by_metric.items():
            if not vals:
                continue

            avg = sum(vals) / len(vals)
            pass_rate = sum(1 for v in vals if v >= self.config.eval_threshold) / len(vals)

            summary["metrics"][name] = {
                "avg_score": round(avg, 4),
                "pass_rate": round(pass_rate, 4),
                "n": len(vals),
            }

        # Save
        if self.scores_file:
            with open(self.scores_file, 'w') as f:
                json.dump(summary, f, indent=2)

        return summary


# ============================================================
# MAIN
# ============================================================

def main():
    """Main entry point."""
    # Parse args
    import argparse
    parser = argparse.ArgumentParser(description="Baseline Evaluator")
    parser.add_argument("--config", type=Path, help="Config file (JSON/YAML)")
    parser.add_argument("--data-dir", type=Path, help="Data directory")
    parser.add_argument("--eval-n-samples", type=int, help="Number of eval samples")
    parser.add_argument("--output-dir", type=Path, help="Output directory")
    parser.add_argument("--log-level", default="INFO", help="Logging level")
    args = parser.parse_args()

    # Load config
    if args.config:
        config = Config.from_file(args.config)
    else:
        config = Config()

    # Override with CLI args
    if args.data_dir:
        config.data_dir = args.data_dir
    if args.eval_n_samples:
        config.eval_n_samples = args.eval_n_samples
    if args.output-dir:
        config.base_dir = args.output_dir

    # Setup logging
    logger = setup_logging(config.base_dir / "logs", getattr(logging, args.log_level))

    # Run
    evaluator = BaselineEvaluator(config, logger)
    summary = evaluator.run()

    logger.info("\n✓ Evaluation complete!")
    return summary


if __name__ == "__main__":
    main()

main()

ModuleNotFoundError: No module named 'litellm'